In [1]:
mock_indexed_blocks = [
    {"id": 1, "text": "[1] <div><p>Advertisement: ...</p></div>"},
    
    {"id": 2, "text": "[2] <h1>AI Helps Humans</h1>"},
    
    {"id": 3, "text": "[3] <p>AI can assist humans in many fields.</p>"},
    
    {"id": 4, "text": "[4] <ul><li>Healthcare: ...</li></ul>"},
    
    {"id": 5, "text": "[5] <li>Education: ...</li>"},
    
    {"id": 6, "text": "[6] <div><h2>Related Articles</h2></div>"},
    
    {"id": 7, "text": "[7] <p>The Future of Robotics</p>"}
]

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# load api keys from .env file
try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
except ImportError:
    pass

from main import predict_relevant_block_ids

query = "What are practical ways AI helps people?"

selected_ids = predict_relevant_block_ids(
    query=query,
    indexed_blocks=mock_indexed_blocks,
    model="qwen-flash",
)

print("query:", query)
print("selected_ids:", selected_ids)


Raw intervals: [[2, 3], [4, 5]]
query: What are practical ways AI helps people?
selected_ids: [2, 3, 4, 5]


In [3]:
from main import reconstruct_html_from_block_ids

reconstructed_html = reconstruct_html_from_block_ids(
    selected_ids=selected_ids,
    indexed_blocks=mock_indexed_blocks,
    keep_page_order=True,
    wrap_container=True,
)

print("reconstructed_html:")
print(reconstructed_html)


reconstructed_html:
<div class="extracted-evidence">
<h1>AI Helps Humans</h1>
<p>AI can assist humans in many fields.</p>
<ul><li>Healthcare: ...</li></ul>
<li>Education: ...</li>
</div>


In [4]:
from main import html_to_markdown

reconstructed_markdown = html_to_markdown(reconstructed_html)

print("reconstructed_markdown:")
print(reconstructed_markdown)

reconstructed_markdown:
# AI Helps Humans

AI can assist humans in many fields.

- Healthcare: ...

- Education: ...


In [ ]:
from main import answer_query_from_extract_results

# ExtractResults can contain multiple webpages, not just one.
extract_results = [
    {
        "doc_id": "doc_1",
        "title": "AI Helps Humans",
        "url": "https://example.com/ai-helps-humans",
        "markdown": reconstructed_markdown,
    },
    {
        "doc_id": "doc_2",
        "title": "AI in Robotics",
        "url": "https://example.com/ai-robotics",
        "markdown": "AI also supports robotics in manufacturing and logistics.",
    },
]

qa_result = answer_query_from_extract_results(
    query=query,
    extract_results=extract_results,
    model="qwen-flash",
)

print("final_answer:", qa_result["answer"])
print("source_ids:", qa_result["source_ids"])
print("source_urls:", qa_result["source_urls"])


final_answer: AI helps people in healthcare, education, manufacturing, and logistics through automation and intelligent decision-making.
source_ids: ['doc_1', 'doc_2']
source_urls: ['https://example.com/ai-helps-humans', 'https://example.com/ai-robotics']
